In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

plt.style.use('sci.mplstyle')

In [2]:
def khoitao_u(dt, dx, tmax, L,f_x, g_x):
    n = int(tmax / dt) + 1   # n la thoi gian
    m = int(L / dx) + 1   # m la khong gian

    u = np.zeros((n, m))
    x = np.linspace(0, L, m)
    t = np.linspace(0, tmax, n)

    # Dieu kien dau theo thoi gian: u(0,x) = f(x), 
    for j in range(m):
        u[0, j] = f_x(x[j], L)
        u[1, j] = u[0, j] + dt * g_x(x[j], L)


    # Dieu kien theo khong gian tai moi thoi diem: u(0,t) = u(L,t) = 0
    u[:, 0] = 0.0
    u[:, -1] = 0.0

    return u, x, t

def tinh_beta(c,k,h):
    beta  = c * k / h
    if beta > 1:
        print("Canh bao: beta =", beta, " > 1. Phuong phap co the khong on dinh.")
        ondinh = False
    else:
        ondinh = True
        print("Gia tri beta =", beta, " <= 1. Phuong phap on dinh.")
        return beta, ondinh

In [3]:
def ham_ghifile(u, x, t, beta, filename, skip_buoc_thoigian):
    filename = f"{filename}_beta_{beta:.3f}.txt"
    with open(filename, "w", encoding="utf-8") as file:
        file.write(f"# Bai toan truyen song 1D \n")
        file.write("# Phuong phap sai phan huu han hien\n")
        file.write("#\n")
        file.write(f"# N_x    = {x.shape[0]}\n")
        file.write(f"# N_time = {t.shape[0]}\n")
        file.write(f"# l      = {x[-1]}\n")
        file.write(f"# t_max  = {t[-1]}\n")
        file.write("#\n")
        file.write(f"# {'t_step':>10s} {'x_step':>10s} {'t':>15s} {'x':>15s} {'u':>15s}\n")

        for t_step in range(t.shape[0]):
            if t_step % skip_buoc_thoigian != 0 and t_step != t.shape[0] - 1: # khi j khong chia het cho skip_buoc_thoigian thi bo qua data do va lay luon  data cuoi cung
                continue
            for x_step in range(x.shape[0]):
                file.write(f"  {t_step:10d} {x_step:10d} {t[t_step]:15.8e} {x[x_step]:15.8e} {u[t_step, x_step]:15.8e}\n")
            file.write("\n\n")
    print("Da luu file voi dia chi: ", filename)

In [4]:
def ham_forward_khong_friction(c,k,h, tmax, L, f_x, g_x, filename, skip_buoc_thoigian=20):
    u, x ,t  = khoitao_u(k, h, tmax, L, f_x, g_x)
    n,m = u.shape
    beta, ondinh = tinh_beta(c, k, h)
    filename = filename + "_no_friction"
    if ondinh == True:
        for i in range(1,n-1): # n la thoi gian
            for j in range(1,m-1): #  m la khong gian
                u[i+1,j] = 2*(1-beta**2)*u[i,j] + beta**2*(u[i,j+1]+u[i,j-1]) - u[i-1,j]
        ham_ghifile(u, x, t, beta, filename, skip_buoc_thoigian)
    else:
        print("Phuong phap khong on dinh, khong the tinh toan.")
    return

# Bai 1, khong friction

In [5]:
## Ham bai 1
def f_x_bai1(x, l):
    if x <= 0.8*l:
        return 1.25*x/l
    else:
        return 5 - 5*x/l

## Van toc bai 1
def g_x_bai1(x, l):
    g_x  = 0
    return g_x

In [6]:
T = 40
rho = 0.01

c = np.sqrt(T/rho)  # Van toc song
print ("Van toc song c =", c)

L               = 1.0           # Chieu dai day
t_max           = 0.05           # Thoi gian mo phong
delta_x         = 0.007          # Buoc khong gian
delta_t         = 0.0001        # Buoc thoi gian  

beta_estimate = c * delta_t / delta_x
print("Beta du kien =", beta_estimate)

Van toc song c = 63.245553203367585
Beta du kien = 0.9035079029052513


In [7]:
ham_forward_khong_friction(c=c, 
                           k=delta_t, 
                           h=delta_x, 
                           tmax=t_max, 
                           L=L, 
                           f_x=f_x_bai1, 
                           g_x=g_x_bai1, 
                           filename="bai1", 
                           skip_buoc_thoigian =5
                           )

Gia tri beta = 0.9035079029052513  <= 1. Phuong phap on dinh.
Da luu file voi dia chi:  bai1_no_friction_beta_0.904.txt


# Bai 2, khong friction

In [8]:
def f_x_bai1_ham_sin(x, l):
    return np.sin(x * np.pi / l)
def g_x_bai1_ham_sin(x, l):
    return 0

In [9]:
ham_forward_khong_friction(c=c, 
                           k=delta_t, 
                           h=delta_x, 
                           tmax=t_max, 
                           L=L, 
                           f_x=f_x_bai1_ham_sin, 
                           g_x=g_x_bai1_ham_sin, 
                           filename="bai2", 
                           skip_buoc_thoigian =5
                           )

Gia tri beta = 0.9035079029052513  <= 1. Phuong phap on dinh.
Da luu file voi dia chi:  bai2_no_friction_beta_0.904.txt
